In [1]:
import os
import glob
import yaml

import pandas as pd
import tifffile
import zarr
import napari
import dask.array as da

from utils.utility_functions import single_channel_pyramid

In [2]:
# I/O

# read single-cell data
main = pd.read_csv(os.path.join(os.getcwd(), 'input/main.csv'))

# read OME-TIFF, segmentation outlines, and H&E channels
tif_path = os.path.join(os.getcwd(), 'input/CyCIF-1A_image.ome.tif')
seg_path = os.path.join(os.getcwd(), 'input/CyCIF-1A_seg_outlines.ome.tif')
he_path = os.path.join(os.getcwd(), 'input/CyCIF-1A_hema_eosin.ome.tif')

# import markers.csv
markers = pd.read_csv(os.path.join(os.getcwd(), 'input/CyCIF-1A_mcmicro_markers.csv'))

# import image contrast settings
with open(os.path.join(os.getcwd(), 'input/CyCIF-1A_cylinter_contrast_limits.yml')) as f:
    contrast_limits = yaml.safe_load(f)

# The parquet file at the path below is being read because "main.csv" 
# uses trimmed marker channel names as column headers that differ from the raw channel names used 
# in the markers.csv file, which is itself used to index channels in the OME-TIFF image.
for_channels = pd.read_parquet(
    os.path.join(os.getcwd(), 'input/CyCIF-1A_clean_cylinter_clustering_3d_leiden.parquet')
)

# isolate antibodies of interest
abx_channels = [i for i in for_channels.columns if 'nucleiRingMask' in i if 'Hoechst' not in i]

In [3]:
# add H&E image to Napari viewer as separate RGB channels
for color, channel in zip(['red', 'green', 'blue'], [0, 1, 2]):

    img, min, max = single_channel_pyramid(glob.glob(he_path)[0], channel=channel)

    if channel == 0:
        viewer = napari.view_image(
            img, rgb=False, colormap=color, blending='additive',
            visible=False, name=f'H&E_{color}', contrast_limits=(min, max)
        )
    else:
        viewer.add_image(
            img, rgb=False, colormap=color, blending='additive',
            visible=False, name=f'H&E_{color}', contrast_limits=(min, max)
        )

In [4]:
# OPTIONAL: add H&E image to Napari viewer as a single channel image

# from lazy_ops import DatasetView
# tiff = tifffile.TiffFile(he_path, is_ome=False)
# pyramid = [
#     zarr.open(tiff.series[0].levels[0].aszarr())[i] for i in
#     list(range(len(tiff.series[0].levels)))
#     ]
# pyramid = [DatasetView(i).lazy_transpose([1, 2, 0]) for i in pyramid]
# pyramid = [da.from_zarr(z) for z in pyramid]
#
# viewer = napari.view_image(pyramid, rgb=True, name='H&E')

In [5]:
# add DNA1 channel to image viewer
dna, min, max = single_channel_pyramid(glob.glob(tif_path)[0], channel=0)
viewer.add_image(
    dna, rgb=False, blending='additive',
    colormap='gray', visible=True, opacity=0.8,
    name='DNA1', contrast_limits=(min, max)
)

<Image layer 'DNA1' at 0x14799fbe0>

In [6]:
# add marker channels to image viewer and apply previously defined contrast limits
for ch in abx_channels:
    ch = ch.rsplit('_', 1)[0]
    channel_number = markers['channel_number'][markers['marker_name'] == ch]
    
    img, min, max = single_channel_pyramid(
        glob.glob(tif_path)[0], channel=(channel_number.item() - 1)
    )
    viewer.add_image(
        img, rgb=False, blending='additive', colormap='lime', visible=False, name=ch,
        contrast_limits=(min, max)
    )
for ch in abx_channels:
    ch = ch.rsplit('_', 1)[0]
    viewer.layers[ch].contrast_limits = (
        contrast_limits[ch][0], contrast_limits[ch][1])

In [7]:
centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 2)]
viewer.add_points(
    centroids, name='Seg3_V2', face_color='#ff7f0e', border_color='white',
    border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
)


centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 7)]
viewer.add_points(
    centroids, name='Seg3_7', face_color='#ff9896', border_color='white',
    border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
)


centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 12)]
viewer.add_points(
    centroids, name='S3_V12', face_color='#e377c2', border_color='white',
    border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
)

centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 14)]
viewer.add_points(
    centroids, name='S3_V14', face_color='#55aaff', border_color='white',
    border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
)


centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 16)]
viewer.add_points(
    centroids, name='Seg3_V16', face_color='#bcbd22', border_color='white',
    border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
)

--Return--
None
> /var/folders/_h/pbzrx8ss6n5f031pf4hc97_w0000gp/T/ipykernel_35451/2313388946.py(29)<module>()
     27 
     28 centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 16)]
---> 29 import pdb; pdb.set_trace()
     30 viewer.add_points(
     31     centroids, name='Seg3_V16', face_color='#bcbd22', border_color='white',



ipdb>  main[main['Seg'] == 3]


         CellID    X_centroid    Y_centroid  Seg   Seg_emb1  Seg_emb2  \
0        142017   3060.428571   8682.897959    3   9.670321  2.843191   
1       1018712  11085.425926  21962.518519    3  15.774168  2.958019   
39       208850  12888.520000  10474.040000    3  15.707607 -1.022952   
48      1100237  14865.914894  22972.297872    3  14.379980 -1.250727   
67      1076461  18855.222222  22682.842593    3  15.206994  3.747733   
...         ...           ...           ...  ...        ...       ...   
479046   876388   4434.452381  20197.976190    3  15.783866  2.956057   
479050  1216073   3115.159420  25052.101449    3  14.296792  0.832447   
479066   530504   8577.714286  15346.857143    3  14.580152  2.132068   
479077   479957  21091.457143  14629.271429    3  13.708219  4.084971   
479083   287292   2562.252632  11955.431579    3  14.361667 -0.784277   

        Seg_emb3  VAE9_VIG7  VAE9_VIG7_emb1  VAE9_VIG7_emb2  ...  CDX2_647  \
0      -1.812012          2        0.081184  

ipdb>  main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 16)]


          Y_centroid    X_centroid
72       7687.943662  19010.112676
111      7889.258065  21833.274194
118      8913.800000  22055.925000
198     13969.214286  21410.757143
214      7383.107692  18565.000000
...              ...           ...
478782   8344.228571  17297.085714
478877   7304.030612  20758.836735
478916   8006.522727  20441.454545
478948   8414.539474  20115.513158
479077  14629.271429  21091.457143

[3714 rows x 2 columns]


...

ipdb>  main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 2)]


          Y_centroid    X_centroid
0        8682.897959   3060.428571
73      14163.975000  15740.700000
95       6978.830189  15431.245283
121     25732.685185   5245.074074
131      5697.438596   4101.684211
...              ...           ...
478899  21512.914286  20157.914286
479029   9968.333333   1351.523810
479046  20197.976190   4434.452381
479050  25052.101449   3115.159420
479066  15346.857143   8577.714286

[7323 rows x 2 columns]


ipdb>  (7323/31694)*100


23.105319618855304


ipdb>  exit


In [ ]:
# add segmentation outlines to image viewer
seg, min, max = single_channel_pyramid(glob.glob(seg_path)[0], channel=0)
viewer.add_image(
    seg, rgb=False, blending='additive',
    colormap='gray', visible=False,
    name='segmentation', opacity=0.3, contrast_limits=(min, max)
)

In [ ]:
# run image viewer
viewer.scale_bar.visible = True
viewer.scale_bar.unit = 'um'